In [ ]:
%matplotlib widget
import pandas 
import pathlib
import ipywidgets
from ipyfilechooser import FileChooser

import numpy

import read_h5

import magic_graphs
import scipp
import scippnexus
import plopp
import widget_code_cell
from ui import spinner

import operations_with_da

In [ ]:
# Global
DF_FILES = pandas.DataFrame(columns=["filename", "size_gb", "modified", "full_path"])

DG_DATA = scipp.DataGroup()

DG_CALC = scipp.DataGroup()

MAGIC_GRAPH = {**magic_graphs.graph_detector, **magic_graphs.graph_qvec, **magic_graphs.graph_hkl}

D_DETECTOR_KEYS = {'da': 'detector_a', 'db': 'detector_b'}

SPATIAL_AXIS_NAMES = ["3D Laue", "Q Space", "HKL Space"]

S_SPATIAL_AXIS_NAMES = ["event_position_global", "Q_vec_rot", "hkl_vec"]

AXIS_NAMES = ["None", "γ", "ν", "TOF", "Q", "2θ", "Q_x", "Q_y", "Q_z", "h", "k", "l", "h_reduced", "k_reduced", "l_reduced", ]

S_AXIS_NAMES = ["", "event_gamma", "event_nu", "toa", "norm_Q", "two_theta", "Q_x", "Q_y", "Q_z", "h", "k", "l", "h_reduced", "k_reduced", "l_reduced", ] # Names like in MAGiC graph




In [ ]:
# Functions
def file_metadata(path: pathlib.Path):
    return {
        "filename": path.name,
        "size_gb": round(path.stat().st_size / 1e9, 2),
        "modified": pandas.to_datetime(path.stat().st_mtime, unit="s"),
        "full_path": path.resolve(),
    }


def display_center(*graphs):
    return display(
        ipywidgets.VBox(graphs, layout=ipywidgets.Layout(align_items="center"))
    )


def range_block(axis_label):
    """Create one range block with min/max/step."""
    
    min_val = ipywidgets.FloatText(
        description="Min:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )
    
    max_val = ipywidgets.FloatText(
        description="Max:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )
    
    step_val = ipywidgets.FloatText(
        description="Step:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )
    
    box = ipywidgets.VBox([
        ipywidgets.HTML(f"<b>{axis_label}</b>"),
        ipywidgets.HBox([min_val, max_val, step_val])
    ])
    
    return {
        "widget": box,
        "min": min_val,
        "max": max_val,
        "step": step_val
    }



def spatial_axis_block(axis_label):
    """Create one axis block with dropdown + min/max/step."""
    
    dropdown = ipywidgets.Dropdown(
        options=SPATIAL_AXIS_NAMES,
        description="",
        layout=ipywidgets.Layout(width="250px")
    )
    
    min_x_val = ipywidgets.FloatText(
        description="Min X:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )
    
    max_x_val = ipywidgets.FloatText(
        description="Max X:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )
    
    step_x_val = ipywidgets.FloatText(
        description="Step X:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )

    min_y_val = ipywidgets.FloatText(
        description="Min Y:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )
    
    max_y_val = ipywidgets.FloatText(
        description="Max Y:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )
    
    step_y_val = ipywidgets.FloatText(
        description="Step Y:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )

    min_z_val = ipywidgets.FloatText(
        description="Min Z:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )
    
    max_z_val = ipywidgets.FloatText(
        description="Max Z:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )
    
    step_z_val = ipywidgets.FloatText(
        description="Step Z:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )

    box = ipywidgets.VBox([
        ipywidgets.HTML(f"<b>{axis_label}</b>"),
        dropdown,
        ipywidgets.HBox([min_x_val, max_x_val, step_x_val]),
        ipywidgets.HBox([min_y_val, max_y_val, step_y_val]),
        ipywidgets.HBox([min_z_val, max_z_val, step_z_val]),
    ])
    
    return {
        "widget": box,
        "dropdown": dropdown,
        "min_x": min_x_val, "max_x": max_x_val, "step_x": step_x_val,
        "min_y": min_y_val, "max_y": max_y_val, "step_y": step_y_val,
        "min_z": min_z_val, "max_z": max_z_val, "step_z": step_z_val,
    }


def axis_block(axis_label):
    """Create one axis block with dropdown + min/max/step."""
    
    dropdown = ipywidgets.Dropdown(
        options=AXIS_NAMES,
        description="",
        layout=ipywidgets.Layout(width="250px")
    )
    
    min_val = ipywidgets.FloatText(
        description="Min:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )
    
    max_val = ipywidgets.FloatText(
        description="Max:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )
    
    step_val = ipywidgets.FloatText(
        description="Step:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )
    
    box = ipywidgets.VBox([
        ipywidgets.HTML(f"<b>{axis_label}</b>"),
        dropdown,
        ipywidgets.HBox([min_val, max_val, step_val])
    ])
    
    return {
        "widget": box,
        "dropdown": dropdown,
        "min": min_val,
        "max": max_val,
        "step": step_val
    }


def assign_dg_to_da_coords(dg: scipp.DataGroup, da: scipp.DataArray, prefix:str=""):
    for s_key in dg.keys():
        da.coords[f"{prefix}_{s_key}"] = dg[s_key]

def move_data_from_dg_to_da(dg_magic, data_event):
    assign_dg_to_da_coords(dg_magic['sample'], data_event, prefix="sample")
    data_event.coords['tp_position'] = dg_magic['tp_position']
    data_event.coords['source_position'] = dg_magic['source_position']
    data_event.coords['delta_L'] = dg_magic['delta_L']
    data_event.coords['delta_t'] = dg_magic['delta_t']
    data_event = data_event.transform_coords(
    ("two_theta",), graph={**magic_graphs.graph_qvec, **magic_graphs.graph_detector}
    )
    return


def load_dg_by_df_row(df_row):
    file_path = df_row['full_path']
    if file_path.suffix == ".h5":
        # McStas file
        d_out = read_h5.read_magic_from_nexus(file_path)
    else: # "hdf, nxs":
        d_out = scippnexus.load(file_path)
    s_key = str(df_row.name)
    DG_DATA[s_key] = d_out
    return
    

def take_dg_by_df_row(df_row):
    s_key = str(df_row.name)
    if not(s_key in DG_DATA.keys()):
        print(f"Loading file '{df_row['filename']}'.")
        load_dg_by_df_row(df_row)
    d_out = DG_DATA[s_key]
    return d_out


def take_name_from_dg_by_df_row(df_row, name:str):
    d_out = take_dg_by_df_row(df_row)
    res = d_out.get(name, None)
    return res


def take_dg_calc_by_df_row(df_row):
    s_key = str(df_row.name)
    if not(s_key in DG_CALC.keys()):
        DG_CALC[s_key] = scipp.DataGroup()
    return DG_CALC[s_key]


def take_detector_a_laue_hist_by_df_row(df_row):
    dg_calc = take_dg_calc_by_df_row(df_row)
    s_key = 'detector_a_laue_hist'
    if not(s_key in dg_calc.keys()):
        print(f"Calculating '{s_key:}'.")
        detector_a = take_name_from_dg_by_df_row(df_row, 'detector_a')
        if detector_a is None:
            return None
        detector_a_laue_hist = operations_with_da.da_to_laue_hist(detector_a, factor_border=0.00)
        dg_calc[s_key] = detector_a_laue_hist
    return dg_calc[s_key]


def take_detector_b_laue_hist_by_df_row(df_row):
    dg_calc = take_dg_calc_by_df_row(df_row)
    s_key = 'detector_b_laue_hist'
    if not(s_key in dg_calc.keys()):
        print(f"Calculating '{s_key:}'.")
        detector_b = take_name_from_dg_by_df_row(df_row, 'detector_b')
        if detector_b is None:
            return None
        detector_b_laue_hist = operations_with_da.da_to_laue_hist(detector_b, factor_border=0.00)
        dg_calc[s_key] = detector_b_laue_hist
    return dg_calc[s_key]


def take_detector_a_image_by_df_row(df_row):
    dg_calc = take_dg_calc_by_df_row(df_row)
    s_key = 'detector_a_image'
    if not(s_key in dg_calc.keys()):
        print(f"Calculating '{s_key:}'.")
        detector_a = take_name_from_dg_by_df_row(df_row, 'detector_a')
        detector_a_image = operations_with_da.da_to_2d_hist(detector_a, factor_border=0.00)
        dg_calc[s_key] = detector_a_image
    return dg_calc[s_key]


def take_detector_b_image_by_df_row(df_row):
    dg_calc = take_dg_calc_by_df_row(df_row)
    s_key = 'detector_b_image'
    if not(s_key in dg_calc.keys()):
        print(f"Calculating '{s_key:}'.")
        detector_b = take_name_from_dg_by_df_row(df_row, 'detector_b')
        detector_b_image = operations_with_da.da_to_2d_hist(detector_b, factor_border=0.00)
        dg_calc[s_key] = detector_b_image
    return dg_calc[s_key]



def get_flag_range_and_points_number(range, np_data):
    """range is (min, max, step)
    """
    points_number = 0
    flag_range = range[2] > 0
    val_min = np_data.min()
    val_max = np_data.max()
    if range[0]!=0.:
        val_min = range[0]
    if range[1]!=0.:
        val_max = range[1]
    if flag_range:
        points_number = int(numpy.round((val_max-val_min)/range[2], 0)) + 1
    return flag_range, (val_min, val_max, points_number)



def get_hist_3d(df_row, s_detector, s_normalization, s_ax_x, range_x, s_ax_y, range_y, s_ax_z, range_z):
    """t_range is (min, max, step)
    """
    dg_calc = take_dg_calc_by_df_row(df_row)
    s_key = f"{s_detector:}/{s_normalization:}/{s_ax_x:}/{s_ax_y:}/{s_ax_z:}/hist_3d"
    detector_key = D_DETECTOR_KEYS[s_detector]
    # if not(s_key in dg_calc.keys()):
    print(f"Calculating '{s_key:}'.")    
    dg_magic = take_dg_by_df_row(df_row)
    detector = dg_magic.get(detector_key, None)
    move_data_from_dg_to_da(dg_magic, detector)
    if detector is None:
        return
    hh = detector.transform_coords((s_ax_x, s_ax_y, s_ax_z), graph=MAGIC_GRAPH)
    d_bin = {"bin_ax_x": 120, "bin_ax_y":120, "bin_ax_z":1000}
    for s_ax, range, s_bin in zip([s_ax_x, s_ax_y, s_ax_z], [range_x, range_y, range_z], ["bin_ax_x", "bin_ax_y", "bin_ax_z"]):
        if hh.coords[s_ax].unit=='rad':
            hh = hh.assign_coords(**{s_ax:hh.coords[s_ax].to(unit="deg")})
        elif hh.coords[s_ax].unit=='s':
            hh = hh.assign_coords(**{s_ax:hh.coords[s_ax].to(unit="ms")})
        flag_range, mms = get_flag_range_and_points_number(range, hh.coords[s_ax].values)
        if flag_range:
            d_bin[s_bin] = scipp.linspace(dim=s_ax, start =mms[0],stop=mms[1],num=mms[2], unit=hh.coords[s_ax].unit, endpoint=True)
    hist_3d = hh.hist(**{s_ax_z:d_bin["bin_ax_z"], s_ax_y:d_bin["bin_ax_y"], s_ax_x:d_bin["bin_ax_x"]})
    dg_calc[s_key] = hist_3d
    return dg_calc[s_key]


def get_hist_2d(df_row, s_detector, s_normalization, s_ax_x, range_x, s_ax_y, range_y):
    """t_range is (min, max, step)
    """
    dg_calc = take_dg_calc_by_df_row(df_row)
    s_key = f"{s_detector:}/{s_normalization:}/{s_ax_x:}/{s_ax_y:}/hist_2d"
    detector_key = D_DETECTOR_KEYS[s_detector]
    # if not(s_key in dg_calc.keys()):
    print(f"Calculating '{s_key:}'.")    
    dg_magic = take_dg_by_df_row(df_row)
    detector = dg_magic.get(detector_key, None)
    move_data_from_dg_to_da(dg_magic, detector)
    if detector is None:
        return
    hh = detector.transform_coords((s_ax_x, s_ax_y), graph=MAGIC_GRAPH)
    d_bin = {"bin_ax_x": 1000, "bin_ax_y":120}
    for s_ax, range, s_bin in zip([s_ax_x, s_ax_y], [range_x, range_y], ["bin_ax_x", "bin_ax_y"]):
        if hh.coords[s_ax].unit=='rad':
            hh = hh.assign_coords(**{s_ax:hh.coords[s_ax].to(unit="deg")})
        elif hh.coords[s_ax].unit=='s':
            hh = hh.assign_coords(**{s_ax:hh.coords[s_ax].to(unit="ms")})
        flag_range, mms = get_flag_range_and_points_number(range, hh.coords[s_ax].values)
        if flag_range:
            d_bin[s_bin] = scipp.linspace(dim=s_ax, start =mms[0],stop=mms[1],num=mms[2], unit=hh.coords[s_ax].unit, endpoint=True)
    hist_2d = hh.hist(**{s_ax_y:d_bin["bin_ax_y"], s_ax_x:d_bin["bin_ax_x"]})
    dg_calc[s_key] = hist_2d
    return dg_calc[s_key]


def get_hist_1d(df_row, s_detector, s_normalization, s_ax_x, range_x):
    """range_x is (min, max, step)
    """
    dg_calc = take_dg_calc_by_df_row(df_row)
    s_key = f"{s_detector:}/{s_normalization:}/{s_ax_x:}/hist_1d"
    detector_key = D_DETECTOR_KEYS[s_detector]
    # if not(s_key in dg_calc.keys()):
    print(f"Calculating '{s_key:}'.")    
    dg_magic = take_dg_by_df_row(df_row)
    detector = dg_magic.get(detector_key, None)
    move_data_from_dg_to_da(dg_magic, detector)
    if detector is None:
        return
    hh = detector.transform_coords((s_ax_x,), graph=MAGIC_GRAPH)
    if hh.coords[s_ax_x].unit=='rad':
        hh = hh.assign_coords(**{s_ax_x:hh.coords[s_ax_x].to(unit="deg")})
    elif hh.coords[s_ax_x].unit=='s':
        hh = hh.assign_coords(**{s_ax_x:hh.coords[s_ax_x].to(unit="ms")})
    flag_range, mms = get_flag_range_and_points_number(range_x, hh.coords[s_ax_x].values)
    if flag_range:
        bin_ax_x = scipp.linspace(dim=s_ax_x, start =mms[0],stop=mms[1],num=mms[2], unit=hh.coords[s_ax_x].unit, endpoint=True)
    else:
        bin_ax_x = 1000
    hist_1d = hh.hist(**{s_ax_x:bin_ax_x})
    dg_calc[s_key] = hist_1d
    return dg_calc[s_key]


def take_detector_a_peaks_gamma_nu_toa(df_row):
    dg_calc = take_dg_calc_by_df_row(df_row)
    s_key = 'detector_a_peaks_gamma_nu_toa'
    if not(s_key in dg_calc.keys()):
        print(f"Calculating '{s_key:}'.")
        dg_magic = take_dg_by_df_row(df_row)
        if dg_magic is None:
            return
        detector_a = dg_magic.get('detector_a', None)
        if detector_a is None:
            return
        detector_a_image = take_detector_a_image_by_df_row(df_row)
        np_toa, np_gamma, np_nu, sig_toa, sig_gamma, sig_nu = (
            operations_with_da.find_peaks_hist(detector_a_image, threshold=0.1)
        )
        dg_calc[s_key] = pandas.DataFrame(
            data={
                'toa': np_toa,
                'gamma': np_gamma,
                'nu': np_nu,
                'toa_sigma': sig_toa,
                'gamma_sigma': sig_gamma,
                'nu_sigma': sig_nu,
                })
    return dg_calc[s_key]


def take_detector_b_peaks_gamma_nu_toa(df_row):
    dg_calc = take_dg_calc_by_df_row(df_row)
    s_key = 'detector_b_peaks_gamma_nu_toa'
    if not(s_key in dg_calc.keys()):
        print(f"Calculating '{s_key:}'.")
        dg_magic = take_dg_by_df_row(df_row)
        if dg_magic is None:
            return
        detector_b = dg_magic.get('detector_b', None)
        if detector_b is None:
            return
        detector_b_image = take_detector_b_image_by_df_row(df_row)
        np_toa, np_gamma, np_nu, sig_toa, sig_gamma, sig_nu = (
            operations_with_da.find_peaks_hist(detector_b_image, threshold=0.1)
        )
        dg_calc[s_key] = pandas.DataFrame(
            data={
                'toa': np_toa,
                'gamma': np_gamma,
                'nu': np_nu,
                'toa_sigma': sig_toa,
                'gamma_sigma': sig_gamma,
                'nu_sigma': sig_nu,
                })
    return dg_calc[s_key]


In [ ]:
# Widgets

fc_path_input = FileChooser(
    path = pathlib.Path('.').resolve(),
    # show_only_dirs=True,
    title='',
    layout=ipywidgets.Layout(width="95%"),
    )


load_button = ipywidgets.Button(
    description="Load",
    button_style="info",
    layout=ipywidgets.Layout(width="120px")
)


clean_button = ipywidgets.Button(
    description="Clean All",
    button_style="danger",
    layout=ipywidgets.Layout(width="120px")
)


clean_data_button = ipywidgets.Button(
    description="Data",
    button_style="danger",
    layout=ipywidgets.Layout(width="120px")
)


clean_calc_button = ipywidgets.Button(
    description="Calculations",
    button_style="danger",
    layout=ipywidgets.Layout(width="120px")
)


extra_reduction_checkbox = ipywidgets.Checkbox(
    value=False,
    description="More",
    disabled=False,
    indent=False
)

time_reduction_block = range_block("Time Reduction")
time_reduction_block['widget'].layout.display= "none"

add_output = ipywidgets.Output()


file_selector = ipywidgets.SelectMultiple(
    options=DF_FILES["filename"].tolist(),
    description="",
    layout=ipywidgets.Layout(width="80%", height="200px")
)


filter = ipywidgets.Text(
    value="",
    placeholder="Filter",
    description="",
    layout=ipywidgets.Layout(width="80%")
)


plot_laue_button = ipywidgets.Button(
    description="3D Laue",
    button_style="info",
    layout=ipywidgets.Layout(width="150px")
)


plot_hist_3d_button = ipywidgets.Button(
    description="γ-ν-TOF",
    button_style="info",
    layout=ipywidgets.Layout(width="150px")
)


plot_hist_2d_button = ipywidgets.Button(
    description="TOF-2θ",
    button_style="info",
    layout=ipywidgets.Layout(width="150px")
)

plot_hist_1d_button = ipywidgets.Button(
    description="TOF",
    button_style="info",
    layout=ipywidgets.Layout(width="150px")
)


extra_plots_checkbox = ipywidgets.Checkbox(
    value=False,
    description="More",
    disabled=False,
    indent=False
)


detector_type_radio_button = ipywidgets.RadioButtons(
    options=['Detector A', 'Detector B'],
    layout={'width': '158px', 'display': 'none'},
    description='',
    disabled=False
)

normalization_radio_button = ipywidgets.RadioButtons(
    options=['No Normalization', 'Per Monitor', 'Per Vanadium'],
    layout={'width': '150px', 'display': 'none'},
    description='',
    disabled=False
)


plot_graph_button = ipywidgets.Button(
    description="Display Graph",
    button_style="info",
    layout={'width': '150px', 'display': 'none'},
)

# Predefined axis names

# Build all three axes
axis_x_block = axis_block("Axis X")
axis_y_block = axis_block("Axis Y")
axis_z_block = axis_block("Axis Z")
axis_spatial_block = spatial_axis_block("Event space")

axis_x_block['widget'].layout.display= "none"
axis_y_block['widget'].layout.display= "none"
axis_z_block['widget'].layout.display= "none"
axis_spatial_block['widget'].layout.display= "none"

find_peaks_button = ipywidgets.Button(
    description="Find Peaks",
    button_style="success",
    layout=ipywidgets.Layout(width="150px")
)

load_output = ipywidgets.Output()

In [ ]:
# Functions that requires widgets
def get_s_detector_and_normalization():
    s_detector = "".join([hh[0] for hh in detector_type_radio_button.value.strip().split()]).lower()
    s_normalization = "".join([hh[0] for hh in normalization_radio_button.value.strip().split()]).lower()
    return (s_detector, s_normalization)

def get_s_1d_ax():
    s_ax_x = S_AXIS_NAMES[AXIS_NAMES.index(axis_x_block['dropdown'].value)]
    if s_ax_x == "":
        s_ax_x = "toa"
    return s_ax_x

def get_s_2d_ax():
    s_ax_y = S_AXIS_NAMES[AXIS_NAMES.index(axis_y_block['dropdown'].value)]
    if s_ax_y == "":
        s_ax_y = "two_theta"
    return get_s_1d_ax(), s_ax_y

def get_s_3d_ax():
    s_ax_x = S_AXIS_NAMES[AXIS_NAMES.index(axis_x_block['dropdown'].value)]
    if s_ax_x == "":
        s_ax_x = "event_gamma"
    s_ax_y = S_AXIS_NAMES[AXIS_NAMES.index(axis_y_block['dropdown'].value)]
    if s_ax_y == "":
        s_ax_y = "event_nu"
    s_ax_z = S_AXIS_NAMES[AXIS_NAMES.index(axis_z_block['dropdown'].value)]
    if s_ax_z == "":
        s_ax_z = "toa"
    return s_ax_x, s_ax_y, s_ax_z

def get_range_1d_ax():
    return (axis_x_block['min'].value, axis_x_block['max'].value, axis_x_block['step'].value)

def get_range_2d_ax():
    range_x = get_range_1d_ax()
    return range_x, (axis_y_block['min'].value, axis_y_block['max'].value, axis_y_block['step'].value)

def get_range_3d_ax():
    range_x, range_y = get_range_2d_ax()
    return range_x, range_y, (axis_z_block['min'].value, axis_z_block['max'].value, axis_z_block['step'].value)


In [ ]:
# Callback
def refresh_selector():
    query = (filter.value or "").strip().lower()
    if DF_FILES.empty:
        file_selector.options = []
        return

    visible_indices = [
        idx for idx, name in DF_FILES["filename"].astype(str).items()
        if not query or query in name.lower()
    ]

    file_selector.options = [
        (DF_FILES.loc[idx, "filename"], idx) for idx in visible_indices
    ]


def on_filter_change(change):
    refresh_selector()


def on_add_clicked(b):
    add_output.clear_output()
    with add_output:
        p = pathlib.Path(fc_path_input.selected_path).expanduser()

        if not p.exists():
            print("Path does not exist:", p)
            return

        global DF_FILES

        # Helper: check if filename already exists in df
        def already_in_df(path):
            return path.resolve() in DF_FILES["full_path"].values

        # Directory → add all files
        if p.is_dir():
            files = [f for f in p.glob("*") if f.is_file() and f.suffix in ['.h5', '.nxs', '.hdf']]
            new_rows = []

            for f in files:
                if already_in_df(f):
                    print(f"Skipping (already in table): {f.name}")
                else:
                    new_rows.append(file_metadata(f))

            if new_rows:
                DF_FILES = pandas.concat([DF_FILES, pandas.DataFrame(new_rows)], ignore_index=True)
                print(f"Added {len(new_rows)} new files from directory:", p)
            else:
                print("No new files to add from directory.")

        # Single file → add only if not already present
        elif p.is_file():
            if already_in_df(p):
                print("File already in table:", p.name)
            else:
                DF_FILES = pandas.concat([DF_FILES, pandas.DataFrame([file_metadata(p)])], ignore_index=True)
                print("Added file:", p.name)
        else:
            print("Path is neither file nor directory:", p)
            return

        refresh_selector()


def on_clean_clicked(b):
    global DF_FILES
    add_output.clear_output()   
    load_output.clear_output()

    DF_FILES = pandas.DataFrame(columns=["filename", "size_gb", "modified", "full_path"])
    on_clean_data_clicked(b)
    on_clean_calc_clicked(b)
    refresh_selector()


def on_clean_data_clicked(b):
    add_output.clear_output()   
    load_output.clear_output()
    DG_DATA.clear()


def on_clean_calc_clicked(b):
    add_output.clear_output()   
    load_output.clear_output()
    DG_CALC.clear()


def on_load_clicked(b):
    load_output.clear_output()
    with load_output:
        selected_indices = list(file_selector.value)
        if not selected_indices:
            print("No files selected.")
            return
        selected_rows = DF_FILES.loc[selected_indices]
        print("Files loading...")
        display(spinner)
        spinner.layout.display = "inline-block"

        for _, row in selected_rows.iterrows():
            file_path = row['full_path']
            if not pathlib.Path(file_path).is_file():
                continue
            try:
                load_dg_by_df_row(row)
                print(f" - {row['filename']} (row index: {row.name})")
                print(f"   full_path: {row['full_path']}")
            except Exception as e:
                print(f"Error reading file: {e}")
                continue
        spinner.layout.display = "none"


def on_plot_laue_clicked(b):
    load_output.clear_output()
    with load_output:
        selected_indices = list(file_selector.value)
        if not selected_indices:
            print("No files selected.")
            return
        selected_rows = DF_FILES.loc[selected_indices]
        print("Plotting Laue Pattern...")
        display(spinner)
        spinner.layout.display = "inline-block"
        fig = None
        for _, row in selected_rows.iterrows():
            d_plot = {}
            vmax = 0.
            detector_a_laue_hist = take_detector_a_laue_hist_by_df_row(row)
            if not(detector_a_laue_hist is None):
                d_plot['detector_a'] = detector_a_laue_hist
                vmax = max([vmax, numpy.quantile(detector_a_laue_hist.data.values, 0.9)])
            detector_b_laue_hist = take_detector_b_laue_hist_by_df_row(row)
            if not(detector_b_laue_hist is None):
                d_plot['detector_b'] = detector_b_laue_hist
                vmax = max([vmax, numpy.quantile(detector_b_laue_hist.data.values, 0.9)])

            fig = plopp.scatter3d(
                d_plot,
                pos="event_position_global",
                cbar=True,
                size=0.005,
                opacity=0.75,
                vmax=vmax,
            )
            break
        spinner.layout.display = "none"
        if fig is None:
            return
        display_center(fig)


def on_plot_3d_graph_clicked(b):
    load_output.clear_output()
    fig = None
    with load_output:
        (s_detector, s_normalization) = get_s_detector_and_normalization()
        s_ax_x, s_ax_y, s_ax_z = get_s_3d_ax()
        range_x, range_y, range_z = get_range_3d_ax()

        selected_indices = list(file_selector.value)
        if not selected_indices:
            print("No files selected.")
            return
        selected_rows = DF_FILES.loc[selected_indices]
        print("Plotting 1d Graph...")
        display(spinner)
        spinner.layout.display = "inline-block"
        for _, row in selected_rows.iterrows():
            hist_3d = get_hist_3d(row, s_detector, s_normalization, s_ax_x, range_x, s_ax_y, range_y, s_ax_z, range_z)
            if hist_3d is None:
                continue
            fig = plopp.inspector(
                hist_3d, 
                dim=s_ax_z,
                orientation="vertical",
                logc=False,
                mode="rectangle",
                autoscale=True,
            )
            break

        spinner.layout.display = "none"
        if fig is None:
            return
        display_center(fig)


def on_plot_2d_graph_clicked(b):
    load_output.clear_output()
    fig = None
    with load_output:
        (s_detector, s_normalization) = get_s_detector_and_normalization()
        s_ax_x, s_ax_y = get_s_2d_ax()
        range_x, range_y = get_range_2d_ax()

        selected_indices = list(file_selector.value)
        if not selected_indices:
            print("No files selected.")
            return
        selected_rows = DF_FILES.loc[selected_indices]
        print("Plotting 1d Graph...")
        display(spinner)
        spinner.layout.display = "inline-block"
        for _, row in selected_rows.iterrows():
            hist_2d = get_hist_2d(row, s_detector, s_normalization, s_ax_x, range_x, s_ax_y, range_y)
            if hist_2d is None:
                continue
            fig = plopp.plot(
                hist_2d, 
                coords=[s_ax_x, s_ax_y],
            )
            break

        spinner.layout.display = "none"
        if fig is None:
            return
        display_center(fig)


def on_plot_1d_graph_clicked(b):
    load_output.clear_output()

    with load_output:
        (s_detector, s_normalization) = get_s_detector_and_normalization()
        s_ax_x = get_s_1d_ax()
        range_x = get_range_1d_ax()


        selected_indices = list(file_selector.value)
        if not selected_indices:
            print("No files selected.")
            return
        selected_rows = DF_FILES.loc[selected_indices]
        print("Plotting 1d Graph...")
        display(spinner)
        spinner.layout.display = "inline-block"
        d_plot = {}
        for _, row in selected_rows.iterrows():
            hist_1d = get_hist_1d(row, s_detector, s_normalization, s_ax_x, range_x)
            if hist_1d is None:
                continue
            d_plot[str(row.name)] = hist_1d
        spinner.layout.display = "none"
        fig = plopp.plot(
            d_plot, 
            coords=[s_ax_x, ],
        )            
        display_center(fig)


def toggle_extra_reduction_checkbox(change):
    if change['new']:          # checkbox checked
        time_reduction_block['widget'].layout.display = 'block'
    else:                      # checkbox unchecked
        time_reduction_block['widget'].layout.display= "none"


def toggle_extra_plots_checkbox(change):
    if change['new']:          # checkbox checked
        detector_type_radio_button.layout.display = 'block'
        normalization_radio_button.layout.display = 'block'
        plot_graph_button.layout.display = 'block'
        axis_x_block['widget'].layout.display= "block"
        axis_y_block['widget'].layout.display= "block"
        axis_z_block['widget'].layout.display= "block"
        axis_spatial_block['widget'].layout.display= "block"
    else:                      # checkbox unchecked
        detector_type_radio_button.layout.display = 'none'
        normalization_radio_button.layout.display = 'none'
        plot_graph_button.layout.display = 'none'
        axis_x_block['widget'].layout.display= "none"
        axis_y_block['widget'].layout.display= "none"
        axis_z_block['widget'].layout.display= "none"
        axis_spatial_block['widget'].layout.display= "none"



def on_plot_graph_clicked(b):
    load_output.clear_output()
    with load_output:
        fig = scipp.show_graph({**magic_graphs.graph_detector, **magic_graphs.graph_qvec, **magic_graphs.graph_hkl}, simplified=True)
        display(fig)


def on_find_peaks_clicked(b):
    load_output.clear_output()
    with load_output:
        selected_indices = list(file_selector.value)
        if not selected_indices:
            print("No files selected.")
            return
        selected_rows = DF_FILES.loc[selected_indices]
        print("Searching peaks...")
        display(spinner)
        spinner.layout.display = "inline-block"
        for _, row in selected_rows.iterrows():
            print(f" - File '{row.name:}'")
            df_peaks_gamma_nu_toa = take_detector_a_peaks_gamma_nu_toa(row)
            if not(df_peaks_gamma_nu_toa is None):
                dg_magic = take_dg_by_df_row(row)
                detector_a = dg_magic.get('detector_a', None)
                detector_a = detector_a.transform_coords(
                    ("event_gamma", "event_nu", "event_position_global"),graph=magic_graphs.graph_detector, rename_dims=False,)
                range_sigma = 5
                operations_with_da.assign_event_peak_to_da(
                    detector_a, 
                    df_peaks_gamma_nu_toa["toa"], 
                    df_peaks_gamma_nu_toa["gamma"], 
                    df_peaks_gamma_nu_toa["nu"], 
                    df_peaks_gamma_nu_toa["toa_sigma"], 
                    df_peaks_gamma_nu_toa["gamma_sigma"], 
                    df_peaks_gamma_nu_toa["nu_sigma"], 
                    range_sigma)
                dg_magic['detector_a'] = detector_a
                print(f"   Number of found peaks on detector A is '{df_peaks_gamma_nu_toa["toa"].size:}'")
            
            df_peaks_gamma_nu_toa = take_detector_b_peaks_gamma_nu_toa(row)
            if not(df_peaks_gamma_nu_toa is None):
                dg_magic = take_dg_by_df_row(row)
                detector_b = dg_magic.get('detector_b', None)
                detector_b = detector_b.transform_coords(
                    ("event_gamma", "event_nu", "event_position_global"),graph=magic_graphs.graph_detector, rename_dims=False,)
                range_sigma = 5
                operations_with_da.assign_event_peak_to_da(
                    detector_b, 
                    df_peaks_gamma_nu_toa["toa"], 
                    df_peaks_gamma_nu_toa["gamma"], 
                    df_peaks_gamma_nu_toa["nu"], 
                    df_peaks_gamma_nu_toa["toa_sigma"], 
                    df_peaks_gamma_nu_toa["gamma_sigma"], 
                    df_peaks_gamma_nu_toa["nu_sigma"], 
                    range_sigma)
                dg_magic['detector_b'] = detector_b
                print(f"   Number of found peaks on detector B is '{df_peaks_gamma_nu_toa["toa"].size:}'")

        spinner.layout.display = "none"

# def update_ax_spatial_on_plot_hist_buttons(change):
#     if change['new'] == 'None':
#         plot_laue_button.description = "3D Laue"
#     else:
#         plot_laue_button.description = change['new']


def update_ax_x_on_plot_hist_buttons(change):
    if change['new'] == 'None':
        plot_hist_1d_button.description = "TOF"
        s_2d = "TOF"
        s_3d = "γ"
    else:
        plot_hist_1d_button.description = change['new']
        s_2d = change['new']
        s_3d = change['new']
    plot_hist_2d_button.description = "-".join([s_2d,]+plot_hist_2d_button.description.split('-')[1:])
    plot_hist_3d_button.description = "-".join([s_3d,]+plot_hist_3d_button.description.split('-')[1:])


def update_ax_y_on_plot_hist_buttons(change):
    if change['new'] == 'None':
        s_2d = "2θ"
        s_3d = "ν"
    else:
        s_2d = change['new']
        s_3d = change['new']
    plot_hist_2d_button.description = "-".join(plot_hist_2d_button.description.split('-')[:-1]+[s_2d,])
    l_hh = plot_hist_3d_button.description.split('-')
    l_hh[1] = s_3d
    plot_hist_3d_button.description = "-".join(l_hh)


def update_ax_z_on_plot_hist_buttons(change):
    if change['new'] == 'None':
        s_3d = "TOF"
    else:
        s_3d = change['new']
    plot_hist_3d_button.description = "-".join(plot_hist_3d_button.description.split('-')[:-1]+[s_3d,])


In [ ]:
# Signals

fc_path_input.register_callback(on_add_clicked)
load_button.on_click(on_load_clicked)
clean_button.on_click(on_clean_clicked)
clean_data_button.on_click(on_clean_data_clicked)
clean_calc_button.on_click(on_clean_calc_clicked)
extra_reduction_checkbox.observe(toggle_extra_reduction_checkbox, names='value')

filter.observe(on_filter_change, names='value')
plot_laue_button.on_click(on_plot_laue_clicked)
plot_hist_3d_button.on_click(on_plot_3d_graph_clicked)
plot_hist_2d_button.on_click(on_plot_2d_graph_clicked)
plot_hist_1d_button.on_click(on_plot_1d_graph_clicked)
extra_plots_checkbox.observe(toggle_extra_plots_checkbox, names='value')
plot_graph_button.on_click(on_plot_graph_clicked)
axis_x_block['dropdown'].observe(update_ax_x_on_plot_hist_buttons, names="value")
axis_y_block['dropdown'].observe(update_ax_y_on_plot_hist_buttons, names="value")
axis_z_block['dropdown'].observe(update_ax_z_on_plot_hist_buttons, names="value")
# axis_spatial_block['dropdown'].observe(update_ax_spatial_on_plot_hist_buttons, names="value")
find_peaks_button.on_click(on_find_peaks_clicked)


In [ ]:
# Display

ipywidgets.VBox([
    fc_path_input, 
    add_output,
    ipywidgets.HBox([file_selector, 
                     ipywidgets.VBox([load_button, clean_button, clean_data_button, clean_calc_button, extra_reduction_checkbox]),]),
    filter,
    time_reduction_block['widget'],
    ipywidgets.HBox([ipywidgets.HTML("<b>Graphs: </b>", layout=ipywidgets.Layout(width="150px")), plot_laue_button, plot_hist_3d_button, plot_hist_2d_button, plot_hist_1d_button, extra_plots_checkbox]),
    ipywidgets.HBox([detector_type_radio_button, normalization_radio_button, plot_graph_button]),
    axis_spatial_block["widget"],
    axis_x_block["widget"],
    axis_y_block["widget"],
    axis_z_block["widget"],
    ipywidgets.HBox([ipywidgets.HTML("<b>Operations: </b>", layout=ipywidgets.Layout(width="150px")), find_peaks_button]),
    load_output
])

In [ ]:
widget_code_cell.make_code_cell(width="80%", placeholder="Write Python code here ...\n\nThe follwoing global variables are available:\n - DF_FILES \n - DG_DATA (stores loaded data) \n - DG_CALC (stores calculated results)")